In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv("hypertension_dataset.csv")

target = "Hypertension"

# -----------------------------
# 2. TARGET CLEANING
# -----------------------------
df[target] = df[target].astype(str).str.strip().str.lower()
df[target] = df[target].map({"high": 1, "low": 0})
df = df.dropna(subset=[target])

# -----------------------------
# 3. FEATURES
# -----------------------------
features = [
    "Age", "BMI", "Systolic_BP", "Diastolic_BP",
    "Family_History", "Diabetes", "Smoking_Status",
    "Physical_Activity_Level", "Glucose", "Salt_Intake"
]

df = df[features + [target]]

# -----------------------------
# 4. CLEAN DATA
# -----------------------------
cat_cols = ["Family_History", "Diabetes", "Smoking_Status", "Physical_Activity_Level"]
num_cols = ["Age", "BMI", "Systolic_BP", "Diastolic_BP", "Glucose", "Salt_Intake"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# -----------------------------
# 5. SPLIT
# -----------------------------
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 6. PIPELINE
# -----------------------------
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

model = XGBClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="logloss"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------------
# 7. TRAIN
# -----------------------------
pipeline.fit(X_train, y_train)

# -----------------------------
# 8. SAVE MODEL
# -----------------------------
joblib.dump(pipeline, "hypertension_pipeline.pkl")

print("✅ Model saved successfully!")

# -----------------------------
# 9. LOAD MODEL (FOR USE)
# -----------------------------
loaded_model = joblib.load("hypertension_model.pkl")

# -----------------------------
# 10. PREDICTION
# -----------------------------
sample = pd.DataFrame([{
    "Age": 50,
    "BMI": 28,
    "Systolic_BP": 135,
    "Diastolic_BP": 88,
    "Family_History": "yes",
    "Diabetes": "no",
    "Smoking_Status": "yes",
    "Physical_Activity_Level": "low",
    "Glucose": 120,
    "Salt_Intake": 8
}])

pred = loaded_model.predict(sample)[0]
prob = loaded_model.predict_proba(sample)[0][1]

print("Prediction (0=Low,1=High):", pred)
print("Risk Probability:", round(prob, 4))

✅ Model saved successfully!
Prediction (0=Low,1=High): 1
Risk Probability: 0.7413


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
